In [2]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AdamW, get_linear_schedule_with_warmup
import warnings

# Suppress FutureWarning
warnings.simplefilter(action='ignore', category=FutureWarning)


# Define a custom dataset class for quotes
class QuotesDataset(Dataset):
    def __init__(self, quotes, tokenizer, max_length):
        self.quotes = quotes
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.quotes)

    def __getitem__(self, idx):
        quote = self.quotes[idx]
        encoding = self.tokenizer(quote, truncation=True, max_length=self.max_length, padding="max_length",
                                  return_tensors="pt")
        input_ids = encoding.input_ids
        return {"input_ids": input_ids}


# Load pre-trained GPT-2 model and tokenizer
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

# Set padding token
tokenizer.pad_token = tokenizer.eos_token

# Load quotes dataset
with open('quots.txt', 'r', encoding='utf-8') as file:
    quotes = file.readlines()

# Define hyperparameters and training settings
batch_size = 5
max_length = 32
num_epochs = 1
learning_rate = 5e-5

# Create dataset and data loader
dataset = QuotesDataset(quotes, tokenizer, max_length)
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Move model to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)

# Define optimizer and learning rate scheduler
optimizer = AdamW(model.parameters(), lr=learning_rate)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0,
                                            num_training_steps=len(dataloader) * num_epochs)

# Fine-tune the model
model.train()
for epoch in range(num_epochs):
    total_loss = 0
    for batch in dataloader:
        input_ids = batch["input_ids"].to(device)
        labels = input_ids.clone()
        outputs = model(input_ids, labels=labels)
        loss = outputs.loss
        total_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(dataloader):.4f}")

# Save the fine-tuned model
model.save_pretrained('fine_tuned_gpt2_quotes')
tokenizer.save_pretrained('fine_tuned_gpt2_quotes')


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Epoch 1/1, Loss: 3.3454


('fine_tuned_gpt2_quotes/tokenizer_config.json',
 'fine_tuned_gpt2_quotes/special_tokens_map.json',
 'fine_tuned_gpt2_quotes/vocab.json',
 'fine_tuned_gpt2_quotes/merges.txt',
 'fine_tuned_gpt2_quotes/added_tokens.json')

In [12]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import torch

# Load the fine-tuned GPT-2 model
model = GPT2LMHeadModel.from_pretrained('fine_tuned_gpt2_quotes')
tokenizer = GPT2Tokenizer.from_pretrained('fine_tuned_gpt2_quotes')

# Function to generate quotes
def generate_quotes(prompt, max_length=100, num_quotes=5):
    quotes = []
    for _ in range(num_quotes):
        input_ids = tokenizer.encode(prompt, return_tensors='pt')
        attention_mask = torch.ones(input_ids.shape, dtype=torch.long)  # Set attention mask to all ones
        output = model.generate(input_ids, attention_mask=attention_mask, max_length=max_length, early_stopping=True)
        generated_quote = tokenizer.decode(output[0], skip_special_tokens=True)
        quotes.append(generated_quote)
    return quotes

# Example prompt
prompt = "Love"

# Generate quotes
generated_quotes = generate_quotes(prompt)
for i, quote in enumerate(generated_quotes, start=1):
    print(f"Generated Quote {i}: {quote}")



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Generated Quote 1: Love.
Generated Quote 2: Love.
Generated Quote 3: Love.
Generated Quote 4: Love.
Generated Quote 5: Love.
